# Detecção e Segmentação de Fauna Silvestre em Rodovias
Notebook executável — clona o repositório do GitHub e roda o pipeline completo (Fases 2 a 4) com GPU.

**Antes de rodar:** Runtime → Change runtime type → GPU (T4 é suficiente).

In [ ]:
# Substitua pela URL real do repositório do grupo
REPO_URL = "https://github.com/SEU_GRUPO/wildlife-roadkill-cv.git"
!git clone $REPO_URL
%cd wildlife-roadkill-cv
!pip install -q -r requirements.txt

## Fase 1/2 — Dataset
Baixe o export do dataset (Roboflow: formato YOLOv8) e extraia em `data/raw/`.
Se estiver no Google Drive, monte o Drive antes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Exemplo: copie o dataset já baixado do Drive para data/raw
# !cp -r /content/drive/MyDrive/dataset_fauna/* data/raw/

### Coleta (iNaturalist) e pré-anotação automática (YOLOE-26)
Baixa fotos reais com licença CC e gera pseudo-labels (caixa + máscara). Pule estas duas células se o grupo já tiver um export anotado em `data/raw/` e `data/raw_seg/`.

In [ ]:
!python scripts/00_download_inaturalist.py --out_dir data/inaturalist --per_class 180
!mkdir -p models && wget -q -nc -P models https://github.com/ultralytics/assets/releases/download/v8.4.0/yoloe-26l-seg.pt
!python scripts/00b_autolabel_yoloe.py --model models/yoloe-26l-seg.pt --overwrite

In [ ]:
!python scripts/01_prepare_dataset.py --raw_dir data/raw --out_dir data/processed --seed 42
!python scripts/01_prepare_dataset.py --raw_dir data/raw_seg --out_dir data/processed --seed 42 --task segment

### EDA — distribuição de classes
Visualize o gráfico gerado (evidência de desbalanceamento para o relatório).

In [ ]:
from IPython.display import Image
Image(filename='data/processed/eda/eda_class_distribution.png')

## Fase 2 — Baseline de detecção (fine-tuning YOLO)

In [ ]:
!python scripts/02_train_detect.py --data config/classes.yaml --epochs 100 --imgsz 640 --batch 16

## Fase 3 — Segmentação de instâncias
Requer labels em formato polygon. Ver `docs/fase1_proposta.md` para o plano de anotação.

In [ ]:
!python scripts/03_train_seg.py --data config/classes_seg.yaml --epochs 100 --imgsz 640 --batch 16

## Fase 4 — Avaliação no conjunto de teste

In [ ]:
!python scripts/04_evaluate.py --weights outputs/detect/weights/best.pt --data config/classes.yaml --split test
!python scripts/04_evaluate.py --weights outputs/segment/weights/best.pt --data config/classes_seg.yaml --split test --name eval_seg --compare_boxes_masks

## Fase 4 — Inferência em vídeo real (obrigatório, min. 30s) + tracking (bônus)
Envie um vídeo real do cenário para `data/video_teste.mp4` antes de rodar.

In [ ]:
!python scripts/05_infer_video.py --weights outputs/segment/weights/best.pt --source data/video_teste.mp4 --track

## Exportar resultados de volta para o Drive (opcional, para não perder ao encerrar a sessão)

In [ ]:
# !cp -r outputs /content/drive/MyDrive/wildlife-roadkill-cv-outputs